# 🍎 Fruit Image Classification with Transfer Learning (ResNet50)

---

## 📌 What You Will Learn

In this notebook, you will:
- Load and preprocess an image dataset using TensorFlow
- Apply **data augmentation** to improve model generalization
- Use **Transfer Learning** with a pre-trained **ResNet50** model
- **Fine-tune** the model for better performance
- Evaluate results and visualize predictions

---

## 🧠 What is Transfer Learning?

Training a deep neural network from scratch requires **millions of images** and days of compute time.

**Transfer Learning** is a smart shortcut: we take a model already trained on a huge dataset (like ImageNet with 1.2 million images) and **re-use its learned features** for our own task.

Think of it like this:
> A doctor who already knows biology doesn't start from zero when learning surgery — they **transfer** existing knowledge.

### Two-Phase Approach We'll Use:
| Phase | What Happens |
|---|---|
| **Phase 1 – Feature Extraction** | Freeze the ResNet50 backbone. Only train the new classification head. |
| **Phase 2 – Fine-Tuning** | Unfreeze the last few layers of ResNet50 and train everything together at a lower learning rate. |

---

## 📁 Dataset Structure

The **Even More Fruits** dataset is already split for us:

```
results/
├── train/
│   ├── Apple/
│   ├── Banana/
│   ├── Cherry/
│   ├── Dragon Fruit/
│   ├── Mango/
│   ├── Orange/
│   ├── Papaya/
│   └── Pineapple/
├── val/
└── test/
```

Each subfolder name **is** the class label. TensorFlow can read this structure automatically!


---
## Step 1 — Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

# Confirm TensorFlow version
print(f"✅ TensorFlow version: {tf.__version__}")

# Check for GPU — training is much faster on GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"🚀 GPU detected: {gpus}")
else:
    print("⚠️  No GPU found — training will run on CPU (slower).")

---
## Step 2 — Configuration

We define all key settings in one place so they're easy to change.

| Parameter | Value | Why? |
|---|---|---|
| `IMG_SIZE` | 224×224 | ResNet50 was designed for this input size |
| `BATCH_SIZE` | 32 | Common default; reduce to 16 if you run out of memory |
| `NUM_CLASSES` | 8 | One per fruit category |
| `SEED` | 42 | Makes random operations reproducible |

In [ ]:
# ── Dataset path ──────────────────────────────────────────────────────────────
# Update this path if your dataset is stored elsewhere
BASE_PATH = '/kaggle/input/datasets/even-more-fruits/results'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VAL_PATH   = os.path.join(BASE_PATH, 'val')
TEST_PATH  = os.path.join(BASE_PATH, 'test')

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE    = (224, 224)   # Required by ResNet50
BATCH_SIZE  = 32           # Lower to 16 if you get OOM errors
NUM_CLASSES = 8            # Apple, Banana, Cherry, Dragon Fruit, Mango, Orange, Papaya, Pineapple
SEED        = 42           # For reproducibility

# ── Training settings ─────────────────────────────────────────────────────────
EPOCHS_PHASE1 = 20         # Feature extraction phase
EPOCHS_PHASE2 = 15         # Fine-tuning phase
LR_PHASE1     = 1e-3       # Higher LR for new layers
LR_PHASE2     = 1e-4       # Lower LR for fine-tuning (avoid destroying pretrained weights)
UNFREEZE_FROM = -30        # Unfreeze last 30 layers of ResNet50 during fine-tuning

print("✅ Configuration set!")
print(f"   Image size : {IMG_SIZE}")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Classes    : {NUM_CLASSES}")

---
## Step 3 — Load the Dataset

`image_dataset_from_directory` is a powerful utility that:
- Reads images from a folder structure
- Automatically assigns integer labels based on subfolder names
- Batches and shuffles data for training
- Returns a `tf.data.Dataset` — optimized for performance

> 💡 **Note:** The dataset is already split into `train/`, `val/`, and `test/` — so we load each folder separately.

In [ ]:
# Load training data — shuffle so the model sees classes in random order
train_data = keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

# Load validation data — no shuffle needed for evaluation
val_data = keras.utils.image_dataset_from_directory(
    VAL_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Load test data — this is our "held-out" final evaluation set
test_data = keras.utils.image_dataset_from_directory(
    TEST_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Class names are inferred from folder names (alphabetical order)
class_names = train_data.class_names
print(f"📂 Classes found ({len(class_names)}): {class_names}")

# Print dataset sizes
print(f"\n📊 Dataset batches:")
print(f"   Train      : {tf.data.experimental.cardinality(train_data).numpy()} batches")
print(f"   Validation : {tf.data.experimental.cardinality(val_data).numpy()} batches")
print(f"   Test       : {tf.data.experimental.cardinality(test_data).numpy()} batches")

---
## Step 4 — Explore the Data

Before training, always **look at your data**. This helps you:
- Confirm images loaded correctly
- Check label assignment
- Get a feel for image quality and variation

In [ ]:
# Grab one batch from training data
sample_images, sample_labels = next(iter(train_data))

print(f"Batch image tensor shape : {sample_images.shape}")
print(f"  → {BATCH_SIZE} images, each {IMG_SIZE[0]}×{IMG_SIZE[1]} px, 3 color channels (RGB)")
print(f"Batch label tensor shape : {sample_labels.shape}")
print(f"Pixel value range        : [{sample_images.numpy().min():.0f}, {sample_images.numpy().max():.0f}]")
print(f"  → Raw images are in [0, 255]; we'll normalize them inside the model")

# Visualize a 4×4 grid of sample images
plt.figure(figsize=(12, 12))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(sample_images[i].numpy().astype('uint8'))
    plt.title(f"{class_names[sample_labels[i]]}", fontsize=10, fontweight='bold')
    plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Step 5 — Performance Optimization with `tf.data`

By default, data loading is done synchronously — the GPU sits idle while the CPU loads the next batch.

**`.cache()`** stores batches in memory after the first epoch, so they load instantly afterward.

**`.prefetch()`** loads the next batch *in the background* while the model trains on the current one — like a conveyor belt!

```
Without prefetch:   Load → Train → Load → Train → ...
With prefetch:      Load+Train → Load+Train → ...
```

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE  # Let TensorFlow decide the optimal buffer size

train_data = train_data.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_data   = val_data.cache().prefetch(buffer_size=AUTOTUNE)
test_data  = test_data.cache().prefetch(buffer_size=AUTOTUNE)

print("✅ Data pipeline optimized with cache() and prefetch()")

---
## Step 6 — Data Augmentation

### What is Data Augmentation?
We artificially **expand** our dataset by applying random transformations to training images. The model sees a slightly different version of each image every epoch — this makes it more **robust** and reduces overfitting.

| Transformation | Purpose |
|---|---|
| `RandomFlip` | The model learns that a flipped fruit is still a fruit |
| `RandomRotation` | Handles fruits at different angles |
| `RandomZoom` | Handles different distances from camera |
| `RandomContrast` | Handles lighting variation |
| `RandomTranslation` | Handles off-center fruits |

> ⚠️ Augmentation is applied **only during training**, not during validation or testing!

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),  # Flip left-right AND up-down
    layers.RandomRotation(0.2),                    # ±20% rotation (≈72°)
    layers.RandomZoom(0.15),                       # ±15% zoom in/out
    layers.RandomContrast(0.2),                    # ±20% contrast adjustment
    layers.RandomTranslation(0.1, 0.1),            # ±10% shift horizontally and vertically
], name='data_augmentation')

print("✅ Augmentation pipeline created!")

# ── Visualize augmentation effect ────────────────────────────────────────────
sample_img = sample_images[0:1]  # Pick first image from our earlier batch

plt.figure(figsize=(12, 4))
plt.subplot(1, 5, 1)
plt.imshow(sample_img[0].numpy().astype('uint8'))
plt.title('Original', fontweight='bold')
plt.axis('off')

for i in range(4):
    augmented = data_augmentation(sample_img, training=True)  # training=True enables randomness
    plt.subplot(1, 5, i + 2)
    plt.imshow(augmented[0].numpy().astype('uint8'))
    plt.title(f'Augmented #{i+1}', fontweight='bold')
    plt.axis('off')

plt.suptitle('Data Augmentation — Same Image, Different Views', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 7 — Build the Model

### ResNet50 Architecture Overview

ResNet50 (Residual Network, 50 layers) introduced **skip connections** — shortcuts that allow gradients to flow directly through earlier layers. This solved the "vanishing gradient" problem that prevented very deep networks from training.

```
Our Model Architecture:

Input (224×224×3)
     ↓
[Data Augmentation]   ← Random transforms during training
     ↓
[Rescaling 0–255 → 0–1]  ← Normalize pixel values
     ↓
[ResNet50 Backbone]   ← Pre-trained on ImageNet (FROZEN in Phase 1)
     ↓
[GlobalAveragePooling2D]  ← Compress feature maps to 1D vector
     ↓
[BatchNorm → Dense(256) → Dropout(0.5)]
     ↓
[BatchNorm → Dense(128) → Dropout(0.3)]
     ↓
[Dense(8, softmax)]   ← 8 output probabilities (one per class)
```

### Why GlobalAveragePooling2D?
Instead of flattening (which creates millions of parameters), GlobalAveragePooling computes the **average of each feature map** — a much more compact and regularized representation.

In [ ]:
# ── Load ResNet50 pre-trained backbone ────────────────────────────────────────
# include_top=False → remove the original ImageNet classification head
# weights='imagenet' → use weights learned from 1.2M images
base_model = keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze ALL layers — we don't want to change ResNet's learned features yet
base_model.trainable = False

frozen_count = sum(1 for layer in base_model.layers if not layer.trainable)
print(f"ResNet50 loaded: {len(base_model.layers)} layers, all {frozen_count} frozen")

# ── Build the full model using the Functional API ─────────────────────────────
inputs = keras.Input(shape=(224, 224, 3), name='input_image')

# Step A: Augment (only active during training, skipped during inference)
x = data_augmentation(inputs)

# Step B: Normalize pixels from [0, 255] to [0, 1]
# ResNet50 expects this range for optimal performance
x = layers.Rescaling(1./255, name='normalize')(x)

# Step C: Pass through frozen ResNet50
# training=False → keeps BatchNorm layers in inference mode (important!)
x = base_model(x, training=False)

# Step D: Compress spatial feature maps → single vector per image
x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)

# Step E: Classification head — our custom layers
x = layers.BatchNormalization(name='bn_1')(x)
x = layers.Dense(256, activation='relu', name='dense_256')(x)
x = layers.Dropout(0.5, name='dropout_1')(x)   # 50% dropout → strong regularization

x = layers.BatchNormalization(name='bn_2')(x)
x = layers.Dense(128, activation='relu', name='dense_128')(x)
x = layers.Dropout(0.3, name='dropout_2')(x)   # 30% dropout → lighter regularization

# Output: 8 neurons (one per class), softmax → probabilities sum to 1
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(x)

model = keras.Model(inputs, outputs, name='ResNet50_FruitClassifier')

# Count trainable vs frozen parameters
total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
frozen_params    = total_params - trainable_params

print(f"\n📐 Model summary:")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable_params:,}  (our classification head)")
print(f"   Frozen parameters    : {frozen_params:,}  (ResNet50 backbone — not updated)")

In [ ]:
# Print the full layer-by-layer summary
model.summary()

---
## Step 8 — Compile the Model

| Setting | Choice | Reason |
|---|---|---|
| **Optimizer** | Adam (lr=1e-3) | Adaptive learning rate — works well out of the box |
| **Loss** | Sparse Categorical Crossentropy | Labels are integers (not one-hot) |
| **Metric** | Accuracy | Intuitive for classification |

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE1),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model compiled!")
print(f"   Optimizer : Adam  (lr={LR_PHASE1})")
print(f"   Loss      : sparse_categorical_crossentropy")
print(f"   Metric    : accuracy")

---
## Step 9 — Set Up Training Callbacks

Callbacks are functions that run **automatically** during training at certain points (end of epoch, etc.).

| Callback | What it does |
|---|---|
| **EarlyStopping** | Stops training if `val_loss` hasn't improved for `patience` epochs — prevents overfitting |
| **ReduceLROnPlateau** | Halves the learning rate when stuck — helps the model escape flat regions |
| **ModelCheckpoint** | Saves the best model to disk automatically |

In [ ]:
# Stop training early if val_loss doesn't improve
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,              # Wait 5 epochs before stopping
    restore_best_weights=True,  # Roll back to the best epoch's weights
    verbose=1
)

# Reduce learning rate when training stalls
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,              # Multiply LR by 0.5
    patience=3,              # Wait 3 epochs before reducing
    min_lr=1e-7,             # Never go below this LR
    verbose=1
)

# Save the best model weights to disk
checkpoint = keras.callbacks.ModelCheckpoint(
    filepath='best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks = [early_stop, reduce_lr, checkpoint]
print("✅ Callbacks ready: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint")

---
## Step 10 — Phase 1: Feature Extraction Training

In this phase:
- The ResNet50 backbone is **frozen** (weights don't change)
- Only the new classification head is trained
- Higher learning rate is safe because we're only training new layers

This phase typically converges quickly (usually < 10 epochs).

In [ ]:
print("=" * 60)
print("  PHASE 1: Feature Extraction (ResNet50 Frozen)")
print("=" * 60)

history_phase1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks
)

print("\n✅ Phase 1 complete!")

# Show best results from Phase 1
best_val_acc = max(history_phase1.history['val_accuracy'])
best_epoch   = history_phase1.history['val_accuracy'].index(best_val_acc) + 1
print(f"   Best validation accuracy: {best_val_acc*100:.2f}% (epoch {best_epoch})")

---
## Step 11 — Phase 2: Fine-Tuning

Now we **unfreeze the last 30 layers** of ResNet50 and train them with a **much lower learning rate**.

### Why a lower learning rate?
The ResNet50 weights are already good — we just want to **gently adjust** them for fruit images, not completely overwrite what they learned from ImageNet.

> ⚠️ Using a high learning rate in fine-tuning would **destroy** the pretrained features!

In [ ]:
print("=" * 60)
print("  PHASE 2: Fine-Tuning (Last 30 ResNet50 Layers Unfrozen)")
print("=" * 60)

# Unfreeze the entire base model first
base_model.trainable = True

# Re-freeze all layers EXCEPT the last 30
for layer in base_model.layers[:UNFREEZE_FROM]:
    layer.trainable = False

# Count newly trainable layers
trainable_layers = [l for l in base_model.layers if l.trainable]
print(f"\n🔓 Unfrozen {len(trainable_layers)} layers in ResNet50 backbone")

# IMPORTANT: Must re-compile after changing trainability
# Use a 10× lower learning rate for fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE2),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"   New learning rate: {LR_PHASE2} (was {LR_PHASE1})\n")

history_phase2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks
)

print("\n✅ Phase 2 complete!")

best_val_acc2 = max(history_phase2.history['val_accuracy'])
best_epoch2   = history_phase2.history['val_accuracy'].index(best_val_acc2) + 1
print(f"   Best validation accuracy: {best_val_acc2*100:.2f}% (epoch {best_epoch2})")

---
## Step 12 — Plot Training Curves

Training curves show us:
- Is the model learning? (loss should decrease)
- Is it overfitting? (big gap between train and val curves = overfitting)
- Did fine-tuning help?

The red dashed line marks where Phase 2 (fine-tuning) began.

In [ ]:
# Combine history from both phases
acc     = history_phase1.history['accuracy']     + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss    = history_phase1.history['loss']         + history_phase2.history['loss']
val_loss= history_phase1.history['val_loss']     + history_phase2.history['val_loss']

phase1_len = len(history_phase1.history['accuracy'])
epochs_range = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Training History — Feature Extraction → Fine-Tuning', 
             fontsize=15, fontweight='bold', y=1.02)

# ── Accuracy plot ─────────────────────────────────────────────────────────────
ax1.plot(epochs_range, [a * 100 for a in acc],     label='Train Accuracy', linewidth=2, color='steelblue')
ax1.plot(epochs_range, [a * 100 for a in val_acc], label='Val Accuracy',   linewidth=2, color='orange', linestyle='--')
ax1.axvline(x=phase1_len, color='red', linestyle=':', linewidth=2, label='Fine-tuning starts')
ax1.fill_between(epochs_range, [a * 100 for a in acc], [a * 100 for a in val_acc],
                 alpha=0.1, color='gray', label='Train–Val gap')
ax1.set_title('Accuracy', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy (%)')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim([0, 100])

# ── Loss plot ─────────────────────────────────────────────────────────────────
ax2.plot(epochs_range, loss,     label='Train Loss', linewidth=2, color='steelblue')
ax2.plot(epochs_range, val_loss, label='Val Loss',   linewidth=2, color='orange', linestyle='--')
ax2.axvline(x=phase1_len, color='red', linestyle=':', linewidth=2, label='Fine-tuning starts')
ax2.set_title('Loss', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Diagnostic interpretation ─────────────────────────────────────────────────
final_gap = abs(acc[-1] - val_acc[-1]) * 100
print("📊 Quick Diagnostics:")
print(f"   Final train accuracy : {acc[-1]*100:.2f}%")
print(f"   Final val accuracy   : {val_acc[-1]*100:.2f}%")
print(f"   Train–Val gap        : {final_gap:.2f}%")
if final_gap > 15:
    print("   ⚠️  Large gap — model may be overfitting. Consider more dropout or more data.")
elif final_gap < 5:
    print("   ✅  Small gap — model generalizes well!")
else:
    print("   ℹ️  Moderate gap — acceptable performance.")

---
## Step 13 — Evaluate on the Test Set

The test set is data the model has **never seen** — not during training, not during validation. This gives us a fair, unbiased measure of real-world performance.

> ⚠️ **Rule:** Only evaluate on the test set **once** — at the very end. Peeking at test results and tuning based on them is called "data leakage" and inflates your metrics.

In [ ]:
print("Evaluating on held-out test set...")
test_loss, test_acc = model.evaluate(test_data, verbose=1)

print(f"\n{'='*40}")
print(f"  Final Test Accuracy : {test_acc*100:.2f}%")
print(f"  Final Test Loss     : {test_loss:.4f}")
print(f"{'='*40}")

---
## Step 14 — Confusion Matrix

A confusion matrix shows exactly **which classes the model confuses with each other**.
- The **diagonal** = correct predictions
- **Off-diagonal** = mistakes (row = true class, column = predicted class)

In [ ]:
# Collect all predictions and true labels from the test set
y_true = []
y_pred = []

for images, labels in test_data:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Build confusion matrix
conf_matrix = tf.math.confusion_matrix(y_true, y_pred, num_classes=NUM_CLASSES).numpy()

# Normalize rows so each cell shows the % of that true class
conf_matrix_norm = conf_matrix.astype('float') / conf_matrix.sum(axis=1, keepdims=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(conf_matrix_norm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax, label='Proportion')

ax.set(xticks=np.arange(NUM_CLASSES),
       yticks=np.arange(NUM_CLASSES),
       xticklabels=class_names,
       yticklabels=class_names,
       ylabel='True Label',
       xlabel='Predicted Label',
       title='Confusion Matrix (Normalized)')

plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=10)

# Add text annotations
thresh = conf_matrix_norm.max() / 2
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f'{conf_matrix_norm[i, j]:.2f}',
                ha='center', va='center',
                color='white' if conf_matrix_norm[i, j] > thresh else 'black',
                fontsize=9)

plt.tight_layout()
plt.show()

# Per-class accuracy
print("\n📊 Per-class accuracy:")
for i, cls in enumerate(class_names):
    per_class_acc = conf_matrix_norm[i, i] * 100
    bar = '█' * int(per_class_acc / 5)
    print(f"   {cls:<15} {bar:<20} {per_class_acc:.1f}%")

---
## Step 15 — Visualize Test Predictions

Let's look at real predictions. **Green title = correct**, **Red title = wrong**.

In [ ]:
# Get one batch of test images
test_images_batch, test_labels_batch = next(iter(test_data))
test_images_np = test_images_batch[:16].numpy()
test_labels_np = test_labels_batch[:16].numpy()

# Get model predictions
predictions = model.predict(test_images_batch[:16], verbose=0)

# Plot predictions
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
axes = axes.flatten()

correct_count = 0
for i in range(16):
    pred_idx    = np.argmax(predictions[i])
    true_idx    = test_labels_np[i]
    confidence  = np.max(predictions[i]) * 100
    pred_class  = class_names[pred_idx]
    true_class  = class_names[true_idx]
    is_correct  = pred_class == true_class
    if is_correct:
        correct_count += 1

    axes[i].imshow(test_images_np[i].astype('uint8'))
    axes[i].set_title(
        f"True: {true_class}\nPred: {pred_class} ({confidence:.1f}%)",
        color='green' if is_correct else 'red',
        fontsize=9, fontweight='bold'
    )
    axes[i].axis('off')

    # Add colored border
    for spine in axes[i].spines.values():
        spine.set_visible(True)
        spine.set_color('green' if is_correct else 'red')
        spine.set_linewidth(3)

green_patch = mpatches.Patch(color='green', label='Correct')
red_patch   = mpatches.Patch(color='red',   label='Incorrect')
fig.legend(handles=[green_patch, red_patch], loc='lower center',
           ncol=2, fontsize=12, bbox_to_anchor=(0.5, -0.01))

plt.suptitle(f'Test Predictions — {correct_count}/16 Correct ({correct_count/16*100:.0f}%)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 16 — Save and Load the Model

In [ ]:
# Save the final trained model
model.save('fruit_classifier_resnet50.keras')
print("✅ Model saved to 'fruit_classifier_resnet50.keras'")

# Load it back (to verify it works)
loaded_model = keras.models.load_model('fruit_classifier_resnet50.keras')
print("✅ Model loaded successfully")

# Confirm loaded model gives same results
sample_preds_original = model.predict(test_images_batch[:4], verbose=0)
sample_preds_loaded   = loaded_model.predict(test_images_batch[:4], verbose=0)
print(f"Predictions match: {np.allclose(sample_preds_original, sample_preds_loaded)}")

---
## 🏁 Summary

Congratulations! You've built a complete image classification pipeline. Here's what you accomplished:

| Step | What You Did |
|------|-------------|
| **1–2** | Set up libraries and configuration |
| **3–5** | Loaded the dataset and optimized the data pipeline |
| **6** | Applied data augmentation to prevent overfitting |
| **7–9** | Built a ResNet50-based model and set up callbacks |
| **10** | Phase 1 — trained only the classification head |
| **11** | Phase 2 — fine-tuned the top ResNet50 layers |
| **12–15** | Evaluated performance with plots, confusion matrix, and visual predictions |
| **16** | Saved and reloaded the model |

---

## 🔧 Challenge Exercises

Try these modifications and observe the effect on accuracy:

1. **Change the backbone**: Replace `ResNet50` with `EfficientNetB0` or `MobileNetV2`. Which is faster? Which is more accurate?
2. **Adjust augmentation**: Remove `RandomFlip` — does accuracy drop on rotated images?
3. **Change the head**: Add a third Dense layer. Does it help or hurt?
4. **Tune dropout**: Set both Dropout rates to 0. What happens to the train–val accuracy gap?
5. **Unfreeze more layers**: Change `UNFREEZE_FROM = -30` to `-50`. Does more fine-tuning help?
6. **Learning rate experiment**: Double `LR_PHASE2` to `2e-4`. Does fine-tuning still work well?

---

## 📚 Key Concepts to Remember

- **Transfer Learning** = reuse knowledge from a large dataset for a smaller task
- **Fine-tuning** = carefully updating pretrained weights with a low learning rate
- **Data Augmentation** = artificially expand training data to improve robustness
- **Early Stopping** = prevent overfitting by stopping when validation loss plateaus
- **`training=False`** in `base_model()` = keep BatchNorm in inference mode during Phase 1
